# Issues
- For vertical levels, sigma-level approximation is used instead of the true hybrid level coordinate system. Pressure levels defined by sigma values are calculated as$$p_{\text{level}} = \sigma_{\text{level}} * p_{\text{surface}}$$ and these $\sigma_{\text{level}}$ are retrieved from [this FAQ page](https://rapidrefresh.noaa.gov/faq/HRRR.faq.html). However, I suspect that this source is outdated as StormCast documentation refers to HRRR's vertical coordinate system as **hybrid** levels and section 2.1 of [this documentation](https://opensky.ucar.edu/islandora/object/technotes%3A576) of Advanced Research WRF model v4 describes a hybrid level system that I suspect HRRR v4 (the latest version) also uses. Anyway I couldn't find a source for the parameters corresponding to each level in the hybrid system so the sigma levels are used as a first attempt approximation.

- HRRR uses Lambert Conformal Grid. However my conversion from ERA5's lat/lon grid to HRRR-format input just bilinearly interpolates the ERA5 grid (something like a spherical coordinate grid) to the required resolution. Again, suitable for a first approximation.

- Composite reflectivity, `refc`, is one of HRRR's 99 variables but does not exist in ERA5. Set to zero everywhere initial conditions for now.

- Some functions are designed to handle multiple initialisation times while others are designed to only handle one (ie assumes that `len(times)==1`). Just stick to one initialisation time for now – I don't quite get the purpose of multiple initialisation times yet.

In [1]:
# ── 0. Install & imports ──────────────────────────────────────────────────────

!pip install -q earth2studio[stormcast]

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import earth2studio.run as run

import utils as ut
import importlib
importlib.reload(ut)

from earth2studio.models.px import StormCast
from earth2studio.data import GFS_FX
from earth2studio.io import ZarrBackend

# ── 1. Configuration ──────────────────────────────────────────────────────────

# File paths
PATH_SURFACE    = "/content/surface_variables.nc"
PATH_PRESSURE   = "/content/pressure_level_variables.nc"
PATH_SP         = "/content/surface_pressure.nc"
PATH_OUTPUT     = "stormcast_output_era5.zarr"

# ── 2. Model & coordinate setup ───────────────────────────────────────────────

package = StormCast.load_default_package()
model   = StormCast.load_model(package, conditioning_data_source=GFS_FX())

coords    = model.input_coords()
VARIABLES = list(coords["variable"])
HRRR_Y    = np.asarray(coords["hrrr_y"])
HRRR_X    = np.asarray(coords["hrrr_x"])
NY, NX    = len(HRRR_Y), len(HRRR_X)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.9/821.9 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 M

In [2]:
# ── 4. Build input array ──────────────────────────────────────────────────────

def build_input_array():
    """Construct the full (1, 99, NY, NX) input DataArray from ERA5 files."""

    ds_surface   = xr.open_dataset(PATH_SURFACE)
    ds_pressure  = xr.open_dataset(PATH_PRESSURE)
    ds_sp        = xr.open_dataset(PATH_SP)

    STARTING_TIME = np.datetime64(ds_surface.valid_time.values[0], "ns")

    src_lats     = ds_surface.latitude.values
    src_lons     = ds_surface.longitude.values
    target_lats  = np.linspace(src_lats.max(), src_lats.min(), NY)  # north → south
    target_lons  = np.linspace(src_lons.min(), src_lons.max(), NX)

    data = xr.DataArray(
        np.zeros((1, len(VARIABLES), NY, NX), dtype=np.float32),
        dims=["time", "variable", "hrrr_y", "hrrr_x"],
        coords={
            "time": [STARTING_TIME],
            "variable": VARIABLES,
            "hrrr_y": HRRR_Y,
            "hrrr_x": HRRR_X,
        },
    )

    # ── 4a. Surface variables (direct mapping) ────────────────────────────────
    for era5_name, sc_name in ut.ERA5_SURFACE_MAP.items():
        raw = ds_surface[era5_name].isel(valid_time=0).values.astype(np.float32)
        field = ut.hinterp(raw, src_lats, src_lons, target_lats, target_lons)

        if np.isnan(field).any():
            raise ValueError(f"NaNs after regridding {era5_name}")

        data.loc[dict(time=STARTING_TIME, variable=sc_name)] = field

        print(
            f"  {era5_name:4s} → {sc_name:5s} | "
            f"min={field.min():.2f}  max={field.max():.2f}"
        )

    # ── 4b. Surface pressure (needed to compute hybrid level pressures) ───────
    sp_raw = ds_sp["sp"].isel(valid_time=0).values.astype(np.float32)
    sp     = ut.hinterp(sp_raw, src_lats, src_lons, target_lats, target_lons)

    print(
        f"\n  sp          | min={sp.min():.1f}  "
        f"max={sp.max():.1f}  mean={sp.mean():.1f} Pa"
    )

    # ── 4c. Pressure-level variables (horizontal regrid then vertical interp) ──────
    era5_p_pa = ds_pressure.pressure_level.values * 100.0   # hPa → Pa

    # Regrid every ERA5 field to the target horizontal grid (done once per variable)
    pressure_3d = {}

    print("\nRegridding ERA5 pressure-level variables...")
    for name in ut.ERA5_PRESSURE_MAP:
        pressure_3d[name] = ut.hinterp_all_levels(
            ds_pressure[name],
            src_lats,
            src_lons,
            target_lats,
            target_lons,
        )
        print(f"  {name} pressure levels regridded")

    for level, sigma in ut.HRRR_SIGMA.items():
        p_target = (sigma * sp).astype(np.float32)      # (NY, NX) target pressure

        # Store the pressure field itself
        p_name = f"p{level}hl"
        if p_name in data["variable"].values:
            data.loc[dict(time=STARTING_TIME, variable=p_name)] = p_target

        # Interpolate each atmospheric variable to this hybrid level
        for era5_name, sc_prefix in ut.ERA5_PRESSURE_MAP.items():
            sc_name = f"{sc_prefix}{level}hl"

            if sc_name not in data["variable"].values:
                continue

            field = ut.vinterp(pressure_3d[era5_name], era5_p_pa, p_target)

            if era5_name == "z":
                field = field / 9.80665    # geopotential [m²/s²] → height [m]

            data.loc[dict(time=STARTING_TIME, variable=sc_name)] = field

    # refc has no ERA5 equivalent — left as zero (placeholder)

    return data, target_lats, target_lons, STARTING_TIME

print("Building input array...")
data, target_lats, target_lons, STARTING_TIME = build_input_array()

is_filled = (data != 0).any(dim=("time", "hrrr_y", "hrrr_x"))

filled = list(data["variable"].values[is_filled.values])
missing = list(data["variable"].values[~is_filled.values])

print(f"\nFilled {len(filled)}/99 variables ({missing} left as zero)")

Building input array...
  t2m  → t2m   | min=288.05  max=301.83
  u10  → u10m  | min=-8.33  max=8.06
  v10  → v10m  | min=-6.74  max=5.00
  msl  → mslp  | min=100767.06  max=101442.03

  sp          | min=82697.3  max=101928.3  mean=100136.0 Pa

Regridding ERA5 pressure-level variables...
  u pressure levels regridded
  v pressure levels regridded
  t pressure levels regridded
  q pressure levels regridded
  z pressure levels regridded

Filled 98/99 variables (['refc'] left as zero)


In [3]:
# ── 5. Wrap and sanity-check ──────────────────────────────────────────────────

my_data = ut.MyLocalData(data)

sample = my_data([STARTING_TIME], VARIABLES)

assert sample.dims == ("time", "variable", "hrrr_y", "hrrr_x")
assert sample.shape == (1, 99, NY, NX)

print(f"Input array verified: {sample.dims} {sample.shape}")

Input array verified: ('time', 'variable', 'hrrr_y', 'hrrr_x') (1, 99, 512, 640)


In [4]:
# ── 6. Inference ──────────────────────────────────────────────────────────────

io = ZarrBackend(PATH_OUTPUT, backend_kwargs={"overwrite": True})
io = run.deterministic(time=[STARTING_TIME], nsteps=2, prognostic=model, data=my_data, io=io)

2026-06-05 02:39:28.016 | INFO     | earth2studio.run:deterministic:78 - Running simple workflow!
2026-06-05 02:39:28.017 | INFO     | earth2studio.run:deterministic:85 - Inference device: cuda
2026-06-05 02:39:29.699 | SUCCESS  | earth2studio.run:deterministic:109 - Fetched data from MyLocalData
2026-06-05 02:39:30.142 | INFO     | earth2studio.run:deterministic:139 - Inference starting!



Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/

2026-06-05 02:39:37.685 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 402200616-968574
2026-06-05 02:39:37.701 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 261515963-813586
2026-06-05 02:39:37.712 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 422644660-945935
2026-06-05 02:39:37.722 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 208633688-736901
2026-06-05 02:39:37.730 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 0-882604
2026-06-05 02:39:37.737 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib fil



Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

2026-06-05 02:39:38.411 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 345504734-941534
2026-06-05 02:39:38.424 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 267790775-926949
2026-06-05 02:39:38.437 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 407574009-960606
2026-06-05 02:39:38.447 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 264301648-1267836
2026-06-05 02:39:38.457 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 409548205-841089
2026-06-05 02:39:38.465 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS

...

2026-06-05 02:40:50.838 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 216861627-594939
2026-06-05 02:40:50.859 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 410607123-1001761
2026-06-05 02:40:50.869 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 267068264-1267148
2026-06-05 02:40:50.877 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 348167163-939282
2026-06-05 02:40:50.886 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 341468135-903401
2026-06-05 02:40:50.895 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GF


Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

2026-06-05 02:40:51.037 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 442311703-1187556
2026-06-05 02:40:51.047 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 271476235-931537
2026-06-05 02:40:51.058 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 402115118-1219434
2026-06-05 02:40:51.067 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 217456566-602156
2026-06-05 02:40:51.077 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 344554143-1238621


Fetching GFS data: 100%|██████████| 26/26 [00:06<00:00,  4.16it/s]

Running inference: 100%|██████████| 3/3 [02:36<00:00, 52.18s/it]

2026-06-05 02:42:06.705 | SUCCESS  | earth2studio.run:deterministic:151 - 
Inference complete


In [5]:
# ── 8. Plots ──────────────────────────────────────────────────────────────────

# Input fields
INPUT_VARS = [
    "t2m", "u10m", "v10m", "mslp",
    "t5hl", "q5hl", "u5hl", "Z5hl", "p5hl",
    "t20hl", "q20hl", "u20hl", "Z20hl",
    "t30hl", "u30hl", "Z30hl", "refc",
]

ut.plot_fields(
    {
        v: data.sel(variable=v).isel(time=0).values
        for v in INPUT_VARS
    },
    title_prefix="Input: ",
)

ut.plot_vertical_profile(data)

# Output fields
ds_out = xr.open_zarr(PATH_OUTPUT)

OUTPUT_VARS = ["t2m", "mslp", "t5hl", "q5hl", "Z5hl", "t20hl", "Z20hl", "refc"]

n_leads = ds_out.sizes["lead_time"]
n_vars = len(OUTPUT_VARS)

fig, axes = plt.subplots(
    n_vars,
    n_leads,
    figsize=(5 * n_leads, 4 * n_vars),
    squeeze=False,
)

for row, var in enumerate(OUTPUT_VARS):
    for col in range(n_leads):
        field = ds_out[var].isel(time=0, lead_time=col).values

        im = axes[row, col].imshow(field, origin="upper")
        axes[row, col].set_title(f"{var} — lead {col}h")
        axes[row, col].set_xlabel("hrrr_x")
        axes[row, col].set_ylabel("hrrr_y")

        plt.colorbar(im, ax=axes[row, col], label=ut.infer_unit(var))

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.